In [ ]:
!pip install -q pandas numpy scikit-learn mlflow fastapi uvicorn joblib httpx

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

def create_dataset(n, monsoon=False):

    humidity = np.random.uniform(45, 95, n)
    temperature = np.random.uniform(16, 32, n)
    wall_temp = temperature - np.random.uniform(0, 5, n)
    door = np.random.randint(0, 12, n)
    ventilation = np.random.uniform(1, 8, n)
    dehumidifier = np.random.uniform(1, 12, n)
    occupancy = np.random.randint(0, 10, n)

    if monsoon:
        humidity = np.clip(humidity + 10, 0, 100)
        ventilation = ventilation * 0.65

    risk = (
        (humidity > 75) |
        ((temperature - wall_temp > 3) & (humidity > 65)) |
        ((door > 7) & (humidity > 70))
    ).astype(int)

    return pd.DataFrame({
        "relative_humidity": humidity,
        "room_temperature": temperature,
        "wall_surface_temperature": wall_temp,
        "door_openings_per_hour": door,
        "ventilation_duration": ventilation,
        "dehumidifier_hours": dehumidifier,
        "room_occupancy": occupancy,
        "moisture_risk": risk
    })

v1 = create_dataset(300)
v2 = create_dataset(300, monsoon=True)

v1.to_csv("dataset_v1.csv", index=False)
v2.to_csv("dataset_v2.csv", index=False)

print("Version 1:", v1.shape)
print("Version 2:", v2.shape)
print("\nDataset created successfully")
display(v1.head())

In [ ]:
data = pd.concat([v1, v2], ignore_index=True)


data = data.fillna(data.median(numeric_only=True))


data["condensation_risk_index"] = (
    data["relative_humidity"] / 100 *
    (data["room_temperature"] -
     data["wall_surface_temperature"])
)

data["ventilation_effectiveness_ratio"] = (
    data["ventilation_duration"] /
    (data["ventilation_duration"] +
     data["dehumidifier_hours"] + 1)
)

# Store features
data.to_csv("feature_store.csv", index=False)

print("Total records:", len(data))
print("Features created successfully")
print("Feature Store saved")
display(data.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

X = data.drop("moisture_risk", axis=1)
y = data["moisture_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight="balanced"
    )
}

results = []
best_model = None
best_recall = -1

for name, model in models.items():

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1
    })

    print("\n", name)
    print("Accuracy:", round(accuracy, 3))
    print("Precision:", round(precision, 3))
    print("Recall:", round(recall, 3))
    print("F1 Score:", round(f1, 3))

    if recall > best_recall:
        best_recall = recall
        best_model = model

results_df = pd.DataFrame(results)

print("\nModel Comparison")
display(results_df)

print("Best Model:", type(best_model).__name__)

In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Manuscript_Moisture_Prediction")

with mlflow.start_run():

    mlflow.log_param(
        "model_name",
        type(best_model).__name__
    )

    mlflow.log_metric("recall", best_recall)

    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="moisture_model",
        serialization_format="pickle",
        registered_model_name="MoistureRiskModel"
    )

print("Model registered successfully in MLflow!")

In [ ]:
import joblib

joblib.dump(best_model, "moisture_model.pkl")
joblib.dump(list(X.columns), "features.pkl")

print("Model saved successfully")


In [ ]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
import pandas as pd
import joblib

app = FastAPI()

model = joblib.load("moisture_model.pkl")
features = joblib.load("features.pkl")

class RoomData(BaseModel):

    relative_humidity: float = Field(ge=0, le=100)
    room_temperature: float = Field(ge=-10, le=60)
    wall_surface_temperature: float
    door_openings_per_hour: int = Field(ge=0)
    ventilation_duration: float = Field(ge=0)
    dehumidifier_hours: float = Field(ge=0)
    room_occupancy: int = Field(ge=0)

@app.get("/health")
def health():
    return {"status": "API is running"}

@app.post("/predict")
def predict(room: RoomData):

    d = room.dict()

    d["condensation_risk_index"] = (
        d["relative_humidity"] / 100 *
        (d["room_temperature"] -
         d["wall_surface_temperature"])
    )

    d["ventilation_effectiveness_ratio"] = (
        d["ventilation_duration"] /
        (d["ventilation_duration"] +
         d["dehumidifier_hours"] + 1)
    )

    df = pd.DataFrame([d])[features]

    result = model.predict(df)[0]

    return {
        "prediction": (
            "Moisture Risk" if result == 1
            else "Storage Conditions Acceptable"
        )
    }

In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)


response = client.get("/health")
print("Health:", response.json())


room = {
    "relative_humidity": 85,
    "room_temperature": 25,
    "wall_surface_temperature": 20,
    "door_openings_per_hour": 3,
    "ventilation_duration": 4,
    "dehumidifier_hours": 5,
    "room_occupancy": 2
}

response = client.post("/predict", json=room)

print("Prediction:", response.json())

# Invalid humidity test
room["relative_humidity"] = 150

response = client.post("/predict", json=room)

print("Invalid Input Status:", response.status_code)

In [ ]:
%%writefile test_app.py

from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

def test_health():
    response = client.get("/health")
    assert response.status_code == 200

def test_prediction():
    data = {
        "relative_humidity": 85,
        "room_temperature": 25,
        "wall_surface_temperature": 20,
        "door_openings_per_hour": 3,
        "ventilation_duration": 4,
        "dehumidifier_hours": 5,
        "room_occupancy": 2
    }

    response = client.post("/predict", json=data)
    assert response.status_code == 200

def test_invalid_humidity():
    data = {
        "relative_humidity": 150,
        "room_temperature": 25,
        "wall_surface_temperature": 20,
        "door_openings_per_hour": 3,
        "ventilation_duration": 4,
        "dehumidifier_hours": 5,
        "room_occupancy": 2
    }

    response = client.post("/predict", json=data)
    assert response.status_code == 422

def test_invalid_temperature():
    data = {
        "relative_humidity": 85,
        "room_temperature": 100,
        "wall_surface_temperature": 20,
        "door_openings_per_hour": 3,
        "ventilation_duration": 4,
        "dehumidifier_hours": 5,
        "room_occupancy": 2
    }

    response = client.post("/predict", json=data)
    assert response.status_code == 422

In [ ]:
!pytest -v test_app.py

In [ ]:
import os

os.makedirs(".github/workflows", exist_ok=True)

In [ ]:
%%writefile .github/workflows/test.yml

name: Moisture Risk CI

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: '3.11'

      - name: Install Libraries
        run: pip install pandas numpy scikit-learn mlflow fastapi uvicorn joblib httpx pytest

      - name: Run Tests
        run: pytest -v

In [ ]:
%%writefile requirements.txt

pandas
numpy
scikit-learn
mlflow
fastapi
uvicorn
joblib
httpx
pytest

In [ ]:
import os

files = [
    "dataset_v1.csv",
    "dataset_v2.csv",
    "feature_store.csv",
    "moisture_model.pkl",
    "features.pkl",
    "app.py",
    "test_app.py",
    "requirements.txt",
    ".github/workflows/test.yml",
    "mlflow.db"
]

print("FINAL PROJECT FILES")

for file in files:
    print("✓", file, "-", os.path.exists(file))

In [ ]:
import subprocess
import time
import requests

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
)

time.sleep(3)

print(requests.get("http://127.0.0.1:8000/health").json())

In [ ]:
server.terminate()

In [ ]:
import subprocess
import time

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
)

time.sleep(5)

print("FastAPI server started")


In [ ]:
from google.colab import output

public_url = output.eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("Your FastAPI URL:", public_url)
print("Health URL:", public_url + "/health")